# Asistente de clasificación de correos / tickets

**Tema:** clasificación de mensajes de soporte en tres categorías — `urgente`, `consulta`, `queja`.

## Idea del proyecto

Comparamos **dos clasificadores** sobre el mismo conjunto de ~100 mensajes etiquetados a mano:

1. **Baseline — Árbol de decisión sobre palabras clave.** Un modelo clásico y
   transparente que decide a partir de la presencia de palabras clave. Lo
   implementamos de dos maneras complementarias:
   - un **árbol de reglas** escrito a mano (if/else legible), y
   - un **`DecisionTreeClassifier` de scikit-learn** entrenado sobre variables
     construidas a partir de esas mismas palabras clave.
2. **LLM.** Un modelo de lenguaje (NVIDIA NIM, vía API compatible con OpenAI)
   al que le pedimos, mediante un *prompt*, que clasifique cada mensaje.

Luego los **medimos con las mismas métricas**: `precision`, `recall` y `F1`
(por clase y promedio *macro*), tal como se hace en problemas de clasificación.

## De qué notebooks reutilizamos conocimiento

- **`ML_exercise.ipynb`** → `DecisionTreeClassifier`, `train_test_split` y la idea
  de que en **clasificación** las métricas correctas son *precision*, *recall* y *F1*
  (no MAE, que es para regresión).
- **`05_ejercicio_nvidia_documentos.ipynb`** → cliente `OpenAI` apuntando a
  **NVIDIA NIM**, lectura de la API key desde `.env` con `python-dotenv`, y
  `client.chat.completions.create(...)` con un *system prompt*.

> Este notebook es **nuevo** y no modifica ninguno de los notebooks del curso.

## Paso 0 — Dependencias

Necesitamos `pandas`, `scikit-learn` (baseline y métricas) y, para el LLM,
`openai` + `python-dotenv` (igual que en `05_ejercicio_nvidia_documentos.ipynb`).

```bash
pip install -q pandas scikit-learn openai python-dotenv matplotlib
```

In [ ]:
# Ejecuta esta celda una sola vez si te faltan dependencias.
# %pip install -q pandas scikit-learn openai python-dotenv matplotlib

## Paso 1 — Cargar el dataset etiquetado a mano

El archivo `tickets_dataset.csv` contiene 100 mensajes de soporte con su
etiqueta (`urgente`, `consulta`, `queja`). Lo cargamos con pandas, igual que en
`ML_exercise.ipynb`.

In [ ]:
import pandas as pd

df = pd.read_csv("tickets_dataset.csv")
print("Filas, columnas:", df.shape)
df.head()

In [ ]:
# Distribución de clases: conviene saber si el dataset está balanceado.
df["etiqueta"].value_counts()

**Observaciones**

- Son 100 mensajes, prácticamente balanceados entre las tres clases (~33 cada una).
  Un dataset balanceado hace que la *accuracy* sea informativa, pero igualmente
  reportaremos *precision*, *recall* y *F1* por clase, que es la práctica correcta
  en clasificación.
- La única columna de texto que usaremos como entrada es `mensaje`; `etiqueta` es
  el *target*.

## Paso 2 — Separar entrenamiento y prueba

Como en `ML_exercise.ipynb`, dividimos en `train`/`test` con `train_test_split`.
Usamos `stratify` para conservar la proporción de clases en ambos conjuntos y
`random_state=0` para reproducibilidad.

El **mismo conjunto de prueba** se usará para evaluar los dos clasificadores, de
modo que la comparación sea justa.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["mensaje"]
y = df["etiqueta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))
print("Clases en prueba:")
print(y_test.value_counts())

## Paso 3 — Baseline (A): árbol de decisión de reglas sobre palabras clave

Primero, la versión **más transparente** del baseline: un árbol de decisión
escrito a mano. La idea es la de cualquier árbol de decisión —una secuencia de
preguntas sí/no— pero aquí las preguntas son sobre **palabras clave**:

```
¿contiene señales de URGENCIA (caído, urgente, crítico, no funciona...)?
├── sí ─► urgente
└── no ─► ¿contiene señales de QUEJA (inaceptable, pésimo, reembolso...)?
         ├── sí ─► queja
         └── no ─► ¿es una pregunta / tiene palabras de CONSULTA (cómo, puedo, tienen...)?
                  ├── sí ─► consulta
                  └── no ─► (desempate por conteo)
```

Normalizamos el texto (minúsculas y sin tildes) para que las palabras clave
coincidan aunque el usuario escriba con o sin acentos.

In [ ]:
import unicodedata

def normaliza(texto):
    # Minúsculas y sin tildes, para comparar palabras clave de forma robusta.
    texto = str(texto).lower()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return texto

# Diccionario de palabras clave por categoría (usamos raíces para cubrir variantes).
KEYWORDS = {
    "urgente": [
        "urgente", "urge", "inmediat", "caid", "fuera de servicio", "no funciona",
        "no responde", "no puedo acceder", "critic", "emergencia", "inaccesible",
        "colaps", "bloquead", "se bloqueo", "fuga de datos", "ataque", "incidente",
        "falla", "fallar", "error 500", "perdemos", "perder datos", "riesgo",
        "alerta", "no accedemos", "no podemos", "se cayo", "se perdio",
        "perdida de conexion", "interrupcion", "corte", "dejo de funcionar",
        "no procesa", "no valida", "no abre", "no permite", "no actualiza",
        "saldos incorrectos", "sobrecalentamiento", "comprometid", "actuen ya",
        "necesitamos ayuda ya", "se cierra sola", "no pueden",
    ],
    "queja": [
        "inaceptable", "pesim", "decepcion", "decepcionad", "lamentable", "harto",
        "frustrante", "estafad", "enganad", "enganar", "enganaron", "malisim",
        "terrible", "nadie me", "nadie resuelve", "no resuelven", "nunca",
        "retraso", "roto", "rompio", "no cumple", "reembolso", "compensacion",
        "quiero mi dinero", "inutil", "decepcionante", "molesto", "grosero",
        "abandonan", "no corresponde", "incompleto", "sin solucion",
        "sin arreglarlo", "sin avisar", "me cobraron", "siguen cobrando",
        "me siento", "me prometieron", "me enviaron", "me cambiaron", "me prometio",
        "mal servicio", "mala imagen", "mala", "no soluciona", "inexistente",
        "no reconozco", "no se presento", "tirado", "sucio", "mas barato",
        "perdiendo la confianza", "pesima experiencia",
    ],
    "consulta": [
        "como", "donde", "cuando", "cuanto", "cual", "cuales", "podrian", "podria",
        "quisiera saber", "me gustaria saber", "me interesa saber", "es posible",
        "puedo", "tienen", "ofrecen", "quisiera", "me gustaria", "disponible",
        "consultar", "diferencia entre", "requisitos", "horario", "queria",
        "puedo obtener", "donde puedo", "cual es", "cuanto tiempo", "que incluye",
        "que metodos",
    ],
}

PRIORIDAD = ["urgente", "queja", "consulta"]

def cuenta_keywords(texto_norm):
    # Cuántas palabras clave de cada categoría aparecen en el texto.
    return {cat: sum(1 for kw in kws if kw in texto_norm) for cat, kws in KEYWORDS.items()}

def arbol_reglas(texto):
    # Árbol de decisión sobre palabras clave, escrito a mano.
    tn = normaliza(texto)
    c = cuenta_keywords(tn)
    es_pregunta = tn.strip().endswith("?")

    # Nodo 1: la urgencia manda.
    if c["urgente"] >= 1 and c["urgente"] >= c["queja"]:
        return "urgente"
    # Nodo 2: señales de queja.
    if c["queja"] >= 1 and c["queja"] >= c["consulta"]:
        return "queja"
    # Nodo 3: pregunta o vocabulario de consulta.
    if es_pregunta or c["consulta"] >= 1:
        return "consulta"
    # Nodo 4: desempate por mayor conteo (por defecto, consulta).
    mejor = max(PRIORIDAD, key=lambda k: c[k])
    return mejor if c[mejor] > 0 else "consulta"

# Prueba rápida
for ejemplo in [
    "El sistema de pagos está caído, necesito solución inmediata",
    "¿Cuál es el horario de atención?",
    "Es inaceptable, llevo semanas esperando el reembolso",
]:
    print(f"{arbol_reglas(ejemplo):9s} <- {ejemplo}")

In [ ]:
# Predicciones del árbol de reglas sobre el conjunto de prueba.
y_pred_reglas = [arbol_reglas(m) for m in X_test]

## Paso 3 — Baseline (B): `DecisionTreeClassifier` sobre variables de palabras clave

Ahora conectamos con `ML_exercise.ipynb` de forma más directa: entrenamos un
**`DecisionTreeClassifier`** de verdad. Un árbol de scikit-learn necesita
variables **numéricas**, así que convertimos cada mensaje en un vector de
conteos de palabras clave (una columna por palabra clave) más una variable
binaria "¿es pregunta?". Es decir, el árbol **aprende** por sí mismo qué
umbrales de qué palabras clave separan mejor las clases.

In [ ]:
import numpy as np

# Lista plana de todas las palabras clave (cada una será una columna / feature).
FEATURE_KWS = [(cat, kw) for cat, kws in KEYWORDS.items() for kw in kws]
FEATURE_NAMES = [f"{cat}:{kw.strip()}" for cat, kw in FEATURE_KWS] + ["es_pregunta"]

def a_features(texto):
    # Convierte un mensaje en un vector numérico de conteos de palabras clave.
    tn = normaliza(texto)
    valores = [tn.count(kw) for _, kw in FEATURE_KWS]
    valores.append(1 if tn.strip().endswith("?") else 0)
    return valores

Xtr = np.array([a_features(m) for m in X_train])
Xte = np.array([a_features(m) for m in X_test])
print("Matriz de entrenamiento:", Xtr.shape, "| prueba:", Xte.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbol = DecisionTreeClassifier(max_depth=6, random_state=0)
arbol.fit(Xtr, y_train)

y_pred_arbol = arbol.predict(Xte)
print("Predicciones del DecisionTreeClassifier generadas.")

### (Opcional) Ver las primeras decisiones del árbol aprendido

`export_text` muestra, en texto, las preguntas que el árbol aprendió. Así se ve
que efectivamente decide a partir de la presencia de palabras clave.

In [ ]:
from sklearn.tree import export_text

print(export_text(arbol, feature_names=FEATURE_NAMES, max_depth=3))

## Paso 4 — Clasificador con LLM (NVIDIA NIM)

Reutilizamos el patrón de `05_ejercicio_nvidia_documentos.ipynb`: un cliente
`OpenAI` apuntando a NVIDIA NIM, con la API key leída de `.env`. La diferencia
es la tarea: en lugar de responder preguntas sobre un documento, le pedimos al
modelo que **clasifique** cada mensaje en una de las tres categorías.

Claves del enfoque:
- Un **system prompt** que define la tarea, las categorías y pide **una sola
  palabra** como respuesta (facilita el parseo).
- `temperature=0` para que la clasificación sea estable y reproducible.
- Una función de **parseo** que normaliza la salida del modelo a una de las tres
  etiquetas válidas.

> Si no hay `NVIDIA_API_KEY` configurada (por ejemplo, sin conexión), la función
> `clasifica_llm` **cae automáticamente** al árbol de reglas para que el notebook
> pueda ejecutarse de principio a fin. Con API key, usa el LLM real.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from openai import OpenAI
    _OPENAI_DISPONIBLE = True
except Exception:
    _OPENAI_DISPONIBLE = False

API_KEY = os.getenv("NVIDIA_API_KEY")

# Modelos candidatos de NVIDIA NIM, en orden de preferencia. Si el primero
# devuelve 404 (no disponible en tu cuenta/catálogo), se prueba el siguiente.
# Puedes ver los nombres válidos en https://build.nvidia.com/ (campo "model"
# del ejemplo de código de cada modelo).
MODELOS_CANDIDATOS = [
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "meta/llama-3.1-8b-instruct",
    "meta/llama-3.3-70b-instruct",
    "mistralai/mistral-small-24b-instruct",
]

client = None
MODELO = None  # se fija al primer modelo que responda correctamente
USAR_LLM = bool(API_KEY) and _OPENAI_DISPONIBLE

if not USAR_LLM:
    print("LLM NO disponible (falta NVIDIA_API_KEY o la librería openai).")
    print("-> clasifica_llm usará el árbol de reglas como respaldo.")
else:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=API_KEY)
    # Probamos cada candidato con una llamada mínima hasta encontrar uno válido.
    for nombre in MODELOS_CANDIDATOS:
        try:
            client.chat.completions.create(
                model=nombre,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=1,
            )
            MODELO = nombre
            print("LLM activado. Modelo:", MODELO)
            break
        except Exception as e:
            print(f"  Modelo no disponible ({nombre}): {type(e).__name__}")
    if MODELO is None:
        USAR_LLM = False
        print("Ningún modelo candidato respondió. Revisa tu API key o ajusta")
        print("MODELOS_CANDIDATOS con un nombre válido de https://build.nvidia.com/.")
        print("-> clasifica_llm usará el árbol de reglas como respaldo.")

In [ ]:
CATEGORIAS = ["urgente", "consulta", "queja"]

SYSTEM_PROMPT = '''Eres un asistente que clasifica mensajes de soporte al cliente.
Debes asignar a cada mensaje EXACTAMENTE una de estas tres categorías:

- urgente: incidentes que requieren acción inmediata; algo está caído, no
  funciona, hay riesgo de seguridad o pérdida, o bloquea la operación.
- consulta: una pregunta o solicitud de información; el cliente quiere saber algo
  y no reporta un fallo ni expresa enfado.
- queja: el cliente expresa insatisfacción o reclama por una mala experiencia,
  aunque no sea un incidente técnico urgente.

Responde ÚNICAMENTE con una palabra en minúsculas: urgente, consulta o queja.
No agregues explicaciones ni signos de puntuación.'''

def parsea_etiqueta(texto):
    # Normaliza la salida del modelo a una de las categorías válidas.
    t = normaliza(texto or "")
    for cat in CATEGORIAS:
        if cat in t:
            return cat
    return "consulta"  # valor por defecto si el modelo responde algo inesperado

def clasifica_llm(mensaje):
    # Clasifica un mensaje con el LLM; si no hay LLM o la llamada falla,
    # usa el árbol de reglas como respaldo para no interrumpir el flujo.
    if not USAR_LLM:
        return arbol_reglas(mensaje)
    try:
        resp = client.chat.completions.create(
            model=MODELO,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": mensaje},
            ],
            temperature=0,
            max_tokens=4,
        )
        return parsea_etiqueta(resp.choices[0].message.content)
    except Exception as e:
        print(f"  [aviso] la llamada al LLM falló ({type(e).__name__}); uso reglas para este mensaje.")
        return arbol_reglas(mensaje)

# Prueba
print(clasifica_llm("El servidor de producción está caído desde hace una hora"))

In [ ]:
# Clasificamos el conjunto de prueba con el LLM (o el respaldo).
# Con API key real esto hace una llamada por mensaje; el test set es pequeño (~30).
y_pred_llm = [clasifica_llm(m) for m in X_test]
print("Clasificación LLM completada sobre", len(y_pred_llm), "mensajes.")

## Paso 5 — Evaluación: precision, recall y F1

Esta es la parte central de la comparación y la conexión directa con lo que ya
conoces: en **clasificación** medimos con *precision*, *recall* y *F1*, no con
MAE (eso es regresión, como se explicaba en `ML_exercise.ipynb`).

- **Precision** (de una clase): de todo lo que el modelo marcó como esa clase,
  ¿qué fracción era correcta?
- **Recall**: de todos los casos que realmente eran de esa clase, ¿qué fracción
  detectó el modelo?
- **F1**: media armónica de precision y recall (equilibra ambas).

Usamos `classification_report` de scikit-learn, que las calcula todas por clase
y además da los promedios *macro* y *weighted*.

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

def evalua(nombre, y_true, y_pred):
    print("=" * 60)
    print("MODELO:", nombre)
    print("=" * 60)
    print(classification_report(y_true, y_pred, labels=CATEGORIAS, digits=3, zero_division=0))
    macro = f1_score(y_true, y_pred, labels=CATEGORIAS, average="macro", zero_division=0)
    print(f"F1 macro: {macro:.3f}")
    return macro

f1_reglas = evalua("Baseline A - Árbol de reglas (keywords)", y_test, y_pred_reglas)
f1_arbol  = evalua("Baseline B - DecisionTreeClassifier (keywords)", y_test, y_pred_arbol)
f1_llm    = evalua("LLM (NVIDIA NIM)" if USAR_LLM else "LLM (respaldo = reglas)", y_test, y_pred_llm)

### Matriz de confusión

La matriz de confusión muestra **dónde** se equivoca cada modelo: filas = clase
real, columnas = clase predicha. La diagonal son los aciertos.

In [ ]:
def muestra_confusion(nombre, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=CATEGORIAS)
    print(nombre)
    print("real \\ pred ", CATEGORIAS)
    for cat, fila in zip(CATEGORIAS, cm):
        print(f"{cat:11s}", fila)
    print()

muestra_confusion("Árbol de reglas:", y_test, y_pred_reglas)
muestra_confusion("DecisionTreeClassifier:", y_test, y_pred_arbol)
muestra_confusion("LLM:", y_test, y_pred_llm)

## Paso 6 — Comparación final

In [ ]:
resumen = pd.DataFrame({
    "modelo": [
        "Árbol de reglas (keywords)",
        "DecisionTreeClassifier (keywords)",
        "LLM (NVIDIA NIM)" if USAR_LLM else "LLM (respaldo = reglas)",
    ],
    "F1_macro": [round(f1_reglas, 3), round(f1_arbol, 3), round(f1_llm, 3)],
})
resumen.sort_values("F1_macro", ascending=False, ignore_index=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.bar(resumen["modelo"], resumen["F1_macro"], color=["#8ecae6", "#219ebc", "#fb8500"])
plt.ylabel("F1 macro")
plt.title("Clasificación de tickets: baseline de keywords vs LLM")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
for i, v in enumerate(resumen["F1_macro"]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

## Conclusiones

- **Clasificación ⇒ precision / recall / F1.** Igual que se señalaba en
  `ML_exercise.ipynb`, al ser un problema de clasificación (no de regresión) las
  métricas adecuadas son *precision*, *recall* y *F1*, y no el MAE.
- **El baseline de palabras clave es rápido, barato y transparente**, pero frágil:
  falla cuando el mensaje describe un incidente o un reclamo **sin usar las
  palabras clave esperadas** (p. ej. "la terminal de punto de venta no procesa
  tarjetas" es urgente, pero puede no contener ninguna keyword de urgencia). La
  matriz de confusión deja ver esas confusiones.
- **El LLM** entiende el contexto y el significado, no solo la coincidencia léxica,
  por lo que suele acertar en los casos que confunden al baseline, a costa de
  latencia y de depender de una API externa.
- **Idea de ingeniería:** se puede combinar ambos — usar el baseline barato para
  los casos claros y reservar el LLM para los dudosos — obteniendo buen
  compromiso entre costo y precisión.

### Para probar con el LLM real

1. Crea un archivo `.env` junto a este notebook con:
   ```text
   NVIDIA_API_KEY="tu_api_key"
   ```
   (Puedes generarla en https://build.nvidia.com/settings/api-keys — igual que en
   `05_ejercicio_nvidia_documentos.ipynb`.)
2. Reinicia el kernel y ejecuta todas las celdas en orden.

> Sin API key, el notebook se ejecuta igualmente: `clasifica_llm` cae al árbol de
> reglas, de modo que la comparación sigue funcionando de principio a fin.